In [ ]:
!pip install pandas scikit-learn transformers torch datasets
# environment for Colab


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/balanced_dataset_30k.csv")
df.head()

In [ ]:
df_distilbert = df

In [ ]:
from sklearn.model_selection import train_test_split

X = df_distilbert["text"]
y = df_distilbert["label"]

X_train_d, X_test_val_d, y_train_d, y_test_val_d = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_test_d, X_val_d, y_test_d, y_val_d = train_test_split(X_test_val_d, y_test_val_d, test_size=0.50, random_state=42, stratify=y_test_val_d)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
from torch.utils.data import Dataset
from torch import tensor


class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        result = self.tokenizer(self.texts[idx], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

        return {"input_ids": result["input_ids"].squeeze(0), "attention_mask": result["attention_mask"].squeeze(0), "label": tensor(self.labels[idx])}

In [ ]:
train_dataset = DepressionDataset(X_train_d.tolist(), y_train_d.tolist(), tokenizer)
test_dataset = DepressionDataset(X_test_d.tolist(), y_test_d.tolist(), tokenizer)
val_dataset = DepressionDataset(X_val_d.tolist(), y_val_d.tolist(), tokenizer)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.to(device)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
model.train()
for epoch in range(3):
    total_loss = 0
    batch_num = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        labels = labels.long()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch + 1}, Batch {batch_num + 1}, Loss: {loss.item():.4f}")
        batch_num += 1

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
all_preds = []
all_labels = []
batch_num = 0
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        batch_num += 1
        print(f"Validation Batch {batch_num} processed.")
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))

In [ ]:
model.eval()
all_preds = []
all_labels = []
batch_num = 0
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        batch_num += 1
        print(f"Validation Batch {batch_num} processed.")
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))

In [ ]:
import pickle
import os

# Define the directory path
output_dir = "/content/drive/MyDrive/Colab Notebooks/models/"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the full path for the model file
model_path = os.path.join(output_dir, "distilbert_model.pkl")

# Save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

In [ ]:
import os

output_dir = "/content/drive/MyDrive/Colab Notebooks/models/"

# List the contents of the directory to confirm the model file is there
print(f"Contents of {output_dir}:")
for item in os.listdir(output_dir):
    print(item)